# ASSIGNMENT - 20 


## PART 1 — DATA PREPROCESSING

### Task 1 — Load and Understand Dataset

Code Cell

In [2]:
import pandas as pd

df = pd.read_csv("dataset/tmdb_5000_movies.csv")

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape:
(4803, 20)

Column Names:
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

First 5 Rows:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictur

Identify Columns

For recommendation we will use:

• title
• overview
• genres
• keywords

These columns contain textual information that can describe the movie.

### Task 2 — Text Preprocessing

Code Cell

In [3]:
import nltk
import re
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Handle Missing Values

In [4]:
df['overview'] = df['overview'].fillna("")

Cleaning Function

In [5]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

Apply Cleaning

In [6]:
df['clean_text'] = df['overview'].apply(clean_text)

df[['title','clean_text']].head()

,title,clean_text
0,Avatar,nd century paraplegic marine dispatched moon p...
1,Pirates of the Caribbean: At World's End,captain barbossa long believed dead come back ...
2,Spectre,cryptic message bonds past sends trail uncover...
3,The Dark Knight Rises,following death district attorney harvey dent ...
4,John Carter,john carter warweary former military captain w...


## PART 2 — TEXT VECTORIZATION

### Task 3 — TF-IDF Vectorization

TF-IDF converts text into numerical vectors so that machine learning models
can process textual data.

We use:
• max_features = 5000
• ngram_range = (1,2)

Code

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(df['clean_text'])

print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

TF-IDF Matrix Shape:
(4803, 5000)


### Task 4 — Similarity Computation

Cosine similarity measures how similar two movie descriptions are.

Why cosine similarity?

• Works well with text vectors
• Measures angle between vectors
• Not affected by document length

Code

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:")
print(similarity_matrix.shape)

Similarity Matrix Shape:
(4803, 4803)


## PART 3 — RECOMMENDATION LOGIC

### Task 5 — Build Recommendation Function

The function:

recommend(movie_name, top_n=5)

Steps:
1 Find index of selected movie
2 Compute similarity scores
3 Sort movies by similarity
4 Return top recommendations

Code

In [9]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

Recommendation Function

In [10]:
def recommend(movie_name, top_n=5):

    if movie_name not in indices:
        return "Movie not found"

    idx = indices[movie_name]

    similarity_scores = list(enumerate(similarity_matrix[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:top_n+1]

    movie_indices = [i[0] for i in similarity_scores]

    return df['title'].iloc[movie_indices]

Test Function

In [11]:
print("Recommendations for Avatar:")
print(recommend("Avatar"))

print("\nRecommendations for Batman Begins:")
print(recommend("Batman Begins"))

print("\nRecommendations for Toy Story:")
print(recommend("Toy Story"))

Recommendations for Avatar:
3604               Apollo 18
2130            The American
1341    The Inhabited Island
529         Tears of the Sun
634               The Matrix
Name: title, dtype: object

Recommendations for Batman Begins:
428                      Batman Returns
4135    Gangster's Paradise: Jerusalema
65                      The Dark Knight
1359                             Batman
3                 The Dark Knight Rises
Name: title, dtype: object

Recommendations for Toy Story:
343                Toy Story 2
42                 Toy Story 3
2869    For Your Consideration
3383                 Losin' It
2479                Radio Days
Name: title, dtype: object


## PART 4 — STREAMLIT APP

Create:
app.py